In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time

spark = (SparkSession.builder
    .appName("UrgenceNet-Partie3-DF")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 22:41:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
event_schema = StructType([
    StructField("visit_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("facility_id", StringType()),
    StructField("city", StringType()),
    StructField("status", StringType()),
    StructField("priority", IntegerType()),
    StructField("wait_minutes", IntegerType()),
    StructField("department", StringType()),
    StructField("desk_id", StringType()),
    StructField("transfer_to", StringType()),
    StructField("event_time", TimestampType()),
])

facility_schema = StructType([
    StructField("facility_id", StringType()),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("hospital_group", StringType()),
    StructField("type", StringType()),
    StructField("beds", IntegerType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
])

ev = spark.read.schema(event_schema).json("s3a://lake/bronze/history/urgences_history.jsonl")
fac = spark.read.schema(facility_schema).json("s3a://lake/bronze/reference/facilities.json")

print(ev.count(), fac.count())
ev.printSchema()
ev.show(3, truncate=False)

26/11/05 22:42:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

50000 40
root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: integer (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- desk_id: string (nullable = true)
 |-- transfer_to: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



+---------+----------+-----------+---------+------------+--------+------------+----------+--------+-----------+-------------------+
|visit_id |patient_id|facility_id|city     |status      |priority|wait_minutes|department|desk_id |transfer_to|event_time         |
+---------+----------+-----------+---------+------------+--------+------------+----------+--------+-----------+-------------------+
|V-0008945|P-06395   |F-029      |Marrakech|ARRIVED     |NULL    |NULL        |Pédiatrie |NULL    |NULL       |2026-10-05 00:22:18|
|V-0008945|P-06395   |F-029      |Marrakech|TRIAGED     |2       |NULL        |Pédiatrie |T-MAR-18|NULL       |2026-10-05 00:30:06|
|V-0008945|P-06395   |F-029      |Marrakech|CARE_STARTED|2       |10          |Pédiatrie |NULL    |NULL       |2026-10-05 00:32:18|
+---------+----------+-----------+---------+------------+--------+------------+----------+--------+-----------+-------------------+
only showing top 3 rows



In [3]:
care = ev.filter(F.col("status") == "CARE_STARTED")

df_res = (care.groupBy("city")
              .agg(F.count("*").alias("prises_en_charge"),
                   F.round(F.avg("wait_minutes"), 2).alias("attente_moy"))
              .orderBy("city"))
df_res.show()

ev.createOrReplaceTempView("events")
spark.sql("""
    SELECT city, COUNT(*) AS prises_en_charge, ROUND(AVG(wait_minutes), 2) AS attente_moy
    FROM events WHERE status = 'CARE_STARTED'
    GROUP BY city ORDER BY city
""").show()

+----------+----------------+-----------+
|      city|prises_en_charge|attente_moy|
+----------+----------------+-----------+
|Casablanca|            2814|      34.86|
| Marrakech|            3331|      34.42|
|     Rabat|            3191|      34.11|
|    Tanger|            3156|      34.96|
+----------+----------------+-----------+



[Stage 12:>                                                         (0 + 1) / 1]

+----------+----------------+-----------+
|      city|prises_en_charge|attente_moy|
+----------+----------------+-----------+
|Casablanca|            2814|      34.86|
| Marrakech|            3331|      34.42|
|     Rabat|            3191|      34.11|
|    Tanger|            3156|      34.96|
+----------+----------------+-----------+



In [4]:
fac.createOrReplaceTempView("facilities")

joined = care.join(fac.select("facility_id", "type", "hospital_group"), "facility_id")
res_join = (joined.groupBy("type", "city")
                  .agg(F.count("*").alias("prises_en_charge"),
                       F.round(F.avg("wait_minutes"), 2).alias("attente_moy"))
                  .orderBy("city", "type"))
res_join.show(20, truncate=False)

+----------------+----------+----------------+-----------+
|type            |city      |prises_en_charge|attente_moy|
+----------------+----------+----------------+-----------+
|CENTRE_SANTE    |Casablanca|209             |30.64      |
|CHU             |Casablanca|960             |41.91      |
|CLINIQUE        |Casablanca|701             |29.87      |
|HOPITAL_REGIONAL|Casablanca|944             |32.33      |
|CENTRE_SANTE    |Marrakech |298             |32.35      |
|CHU             |Marrakech |1059            |40.69      |
|CLINIQUE        |Marrakech |891             |32.34      |
|HOPITAL_REGIONAL|Marrakech |1083            |30.57      |
|CENTRE_SANTE    |Rabat     |263             |34.12      |
|CHU             |Rabat     |1126            |38.58      |
|CLINIQUE        |Rabat     |834             |31.21      |
|HOPITAL_REGIONAL|Rabat     |968             |31.4       |
|CENTRE_SANTE    |Tanger    |252             |32.11      |
|CHU             |Tanger    |951             |39.51     

In [5]:
def wait_band_py(w):
    if w is None:
        return None
    if w < 15:
        return "court"
    if w <= 45:
        return "moyen"
    return "long"

wait_band_udf = F.udf(wait_band_py, StringType())

def temps(df):
    t0 = time.time()
    df.groupBy("band").count().orderBy("band").collect()
    return time.time() - t0

with_udf = care.withColumn("band", wait_band_udf("wait_minutes"))
with_when = care.withColumn("band",
    F.when(F.col("wait_minutes") < 15, "court")
     .when(F.col("wait_minutes") <= 45, "moyen")
     .otherwise("long"))

temps(with_udf); temps(with_when)          # chauffage
print("UDF Python   : %.2f s" % temps(with_udf))
print("when/otherwise: %.2f s" % temps(with_when))

with_when.groupBy("band").count().orderBy("band").show()
with_udf.groupBy("band").count().explain()
with_when.groupBy("band").count().explain()

UDF Python   : 8.83 s


when/otherwise: 4.54 s


+-----+-----+
| band|count|
+-----+-----+
|court| 4840|
| long| 3137|
|moyen| 4515|
+-----+-----+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[band#275], functions=[count(1)])
   +- Exchange hashpartitioning(band#275, 200), ENSURE_REQUIREMENTS, [plan_id=858]
      +- HashAggregate(keys=[band#275], functions=[partial_count(1)])
         +- Project [pythonUDF0#428 AS band#275]
            +- BatchEvalPython [wait_band_py(wait_minutes#6)#274], [pythonUDF0#428]
               +- Project [wait_minutes#6]
                  +- Filter (isnotnull(status#4) AND (status#4 = CARE_STARTED))
                     +- FileScan json [status#4,wait_minutes#6] Batched: false, DataFilters: [isnotnull(status#4), (status#4 = CARE_STARTED)], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/urgences_history.jsonl], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,CARE_STARTED)], ReadSchema: struct<status:string,wait_minutes:i

In [6]:
arr = (ev.filter("status = 'ARRIVED'")
         .dropDuplicates(["visit_id"])
         .select("visit_id", F.col("event_time").alias("t_arr")))
cs = (ev.filter("status = 'CARE_STARTED'")
        .dropDuplicates(["visit_id"])
        .select("visit_id", F.col("event_time").alias("t_care"), "wait_minutes"))

chk = (arr.join(cs, "visit_id")
          .withColumn("attente_calc",
                      F.round((F.col("t_care").cast("long") - F.col("t_arr").cast("long")) / 60).cast("int")))

total = chk.count()
ok = chk.filter(F.col("attente_calc") == F.col("wait_minutes")).count()
print("Passages comparés :", total, "| cohérents :", ok, "| écarts :", total - ok)
chk.filter(F.col("attente_calc") != F.col("wait_minutes")).show(5)

Passages comparés : 12264 | cohérents : 12264 | écarts : 0


[Stage 75:>                                                         (0 + 1) / 1]

+--------+-----+------+------------+------------+
|visit_id|t_arr|t_care|wait_minutes|attente_calc|
+--------+-----+------+------------+------------+
+--------+-----+------+------------+------------+



In [7]:
avant = ev.count()
clean = ev.dropDuplicates(["visit_id", "status"])
apres = clean.count()
print("Avant :", avant, "| Après :", apres, "| Doublons supprimés :", avant - apres)

(clean.write.mode("overwrite")
      .partitionBy("city")
      .parquet("s3a://lake/silver/urgences_history"))

spark.read.parquet("s3a://lake/silver/urgences_history").groupBy("city").count().show()

Avant : 50000 | Après : 49056 | Doublons supprimés : 944


[Stage 89:===========================================>              (3 + 1) / 4]

+----------+-----+
|      city|count|
+----------+-----+
|Casablanca|11012|
|    Tanger|12360|
|     Rabat|12552|
| Marrakech|13132|
+----------+-----+



In [8]:
j = care.join(fac, "facility_id").groupBy("type").count()
j.explain(True)

== Parsed Logical Plan ==
'Aggregate ['type], ['type, count(1) AS count#954L]
+- Project [facility_id#2, visit_id#0, patient_id#1, city#3, status#4, priority#5, wait_minutes#6, department#7, desk_id#8, transfer_to#9, event_time#10, name#23, city#24, hospital_group#25, type#26, beds#27, latitude#28, longitude#29]
   +- Join Inner, (facility_id#2 = facility_id#22)
      :- Filter (status#4 = CARE_STARTED)
      :  +- Relation [visit_id#0,patient_id#1,facility_id#2,city#3,status#4,priority#5,wait_minutes#6,department#7,desk_id#8,transfer_to#9,event_time#10] json
      +- Relation [facility_id#22,name#23,city#24,hospital_group#25,type#26,beds#27,latitude#28,longitude#29] json

== Analyzed Logical Plan ==
type: string, count: bigint
Aggregate [type#26], [type#26, count(1) AS count#954L]
+- Project [facility_id#2, visit_id#0, patient_id#1, city#3, status#4, priority#5, wait_minutes#6, department#7, desk_id#8, transfer_to#9, event_time#10, name#23, city#24, hospital_group#25, type#26, beds#27

In [9]:
def pipeline():
    t0 = time.time()
    (spark.read.schema(event_schema).json("s3a://lake/bronze/history/urgences_history.jsonl")
          .filter("status = 'CARE_STARTED'")
          .groupBy("city")
          .agg(F.count("*"), F.avg("wait_minutes"))
          .collect())
    return time.time() - t0

pipeline()
print("Temps DataFrame : %.2f s" % pipeline())

[Stage 95:=============================>                            (2 + 2) / 4]

Temps DataFrame : 3.57 s


----------------------------------------                                        
Exception occurred during processing of request from ('127.0.0.1', 48078)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/s